# Day model v4 — MAE mean, then P10 / P90 tails

v1–v3 added columns. Shrinking the quantile trees made coverage **53%** (scale stuck at 1.04) while MAE only moved a little.

Same MAE mean, P10/P90 tails, and pre-refit coverage scale as the last v4 run, plus **24h clock-match deltas** (weather vs yesterday, `demand_lag_96`, `demand_delta_1d`). Not v2 EMAs or 7-day extras.

1. Mean / P50 uses LightGBM **MAE** with the unused Optuna **short** winner (`LGB_DAY`).
2. Only **P10 and P90** are quantile trees.
3. `interval_scale` is fitted on the 21-day calib split **before** the full-train refit.

Writes to `notebooks/day/v4/models`. Re-run **train**. `PROMOTE = False`.

In [1]:
from pathlib import Path
import json
import shutil
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "energy_forecast").is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
HERE = ROOT / "notebooks" / "day" / "v4"
OUT_DIR = HERE / "models"
sys.path.insert(0, str(ROOT / "src"))

from energy_forecast import tune
from energy_forecast.evaluate import as_board_scored, board_naive_scale, overall_metrics
from energy_forecast.explain import explain_scored, mean_abs_shap, shap_source_share
from energy_forecast.features import DAY_V4_FEATURES, FEATURE_COLUMNS, attach_lookback_demand, attach_lookback_weather
from energy_forecast.forecast import prepare_history
from energy_forecast.live import DAY, load_board, load_context_actuals
from energy_forecast.model import (
    DAY_MAX_LEAD_HOURS,
    DAY_N_ESTIMATORS,
    LGB_DAY,
    TAIL_QUANTILES,
    load_day_models,
    predict_quantiles,
    split_training_frame,
    train_models,
)
from energy_forecast.modes import SHORT, resample_to_mode
from energy_forecast.regions import SERIES_ID
from energy_forecast.review import (
    _attach_frozen_and_weather,
    _ensure_calendar,
    _load_archive_weather,
    _pack_rows,
    _ready_predict_frame,
    _recompute_weather_features,
)
from energy_forecast.settlement import LONDON_TZ
from energy_forecast.tune import naive_scale_for_product

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
prod = load_day_models()
print("repo", ROOT)
print("write models to", OUT_DIR)
print("features", len(DAY_V4_FEATURES), "production", len(FEATURE_COLUMNS))
print("24h extras", [c for c in DAY_V4_FEATURES if c not in FEATURE_COLUMNS])
print("prod interval_scale", prod.interval_scale)
print("LGB_DAY", LGB_DAY)
print("tail quantiles", TAIL_QUANTILES)

repo D:\energy-forecast
write models to D:\energy-forecast\notebooks/day/v4\models
features 47 production 37
24h extras ['temp_delta_1d', 'temp_mean_24h_delta_1d', 'apparent_temp_delta_1d', 'hdd_delta_1d', 'cdd_delta_1d', 'shortwave_delta_1d', 'wind_delta_1d', 'cloud_delta_1d', 'demand_lag_96', 'demand_delta_1d']
prod interval_scale {'gb': 2.079867720603943}
LGB_DAY {'learning_rate': 0.05154938259471686, 'num_leaves': 45, 'min_child_samples': 62, 'subsample': 0.8084151420110467, 'colsample_bytree': 0.6235476485104684, 'feature_fraction': 0.6235476485104684}
tail quantiles (0.1, 0.9)


## 1. History

Previous Runs, leads ≤ 24h. Same split as production.

In [2]:
history = prepare_history()
train, holdout = split_training_frame(
    history,
    features=DAY_V4_FEATURES,
    max_lead_hours=DAY_MAX_LEAD_HOURS,
)
print("train", len(train), "holdout", len(holdout))

train 91102 holdout 5090


## 2. Optional Optuna

Default is off: use the shipped `short` winner. Set `RUN_OPTUNA = True` if you want another 40 trials on this split.

In [3]:
PRODUCT = "day_v4"
RUN_OPTUNA = True
N_TRIALS = 40
USE_WANDB = False

mean_params = {
    "learning_rate": LGB_DAY["learning_rate"],
    "num_leaves": LGB_DAY["num_leaves"],
    "min_child_samples": LGB_DAY["min_child_samples"],
    "subsample": LGB_DAY["subsample"],
    "colsample_bytree": LGB_DAY["colsample_bytree"],
    "feature_fraction": LGB_DAY["feature_fraction"],
    "n_estimators": DAY_N_ESTIMATORS,
    "objective": "mae",
    "metric": "mae",
}
if RUN_OPTUNA:
    tune_train, tune_holdout = tune.product_split(history, PRODUCT)
    study = tune.create_study(PRODUCT)
    with tune.tracking_run(PRODUCT, study, wandb_on=USE_WANDB):
        study.optimize(tune.make_objective(tune_train, tune_holdout, PRODUCT), n_trials=N_TRIALS, show_progress_bar=True)
        tune.log_study_end(study, PRODUCT)
    mean_params = {**mean_params, **study.best_params}
    mean_params["feature_fraction"] = float(mean_params["colsample_bytree"])
    print("optuna best MAE", study.best_value)
print("mean_params", mean_params)
(HERE / "optuna_best.json").write_text(json.dumps(mean_params, indent=2), encoding="utf-8")

[I 2026-10-09 08:55:29,236] Using an existing study with `study_name='day_v4_mae'` instead of creating a new one.


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-10-09 08:55:48,002] Trial 50 finished with value: 814.7805816473424 and parameters: {'learning_rate': 0.04613392540592092, 'num_leaves': 46, 'min_child_samples': 74, 'subsample': 0.6488524892756541, 'colsample_bytree': 0.5615911847372813, 'n_estimators': 500}. Best is trial 50 with value: 814.7805816473424.
[I 2026-10-09 08:56:02,167] Trial 51 finished with value: 806.0024020395334 and parameters: {'learning_rate': 0.037044963874225616, 'num_leaves': 47, 'min_child_samples': 72, 'subsample': 0.6463298629927613, 'colsample_bytree': 0.5768447192053281, 'n_estimators': 450}. Best is trial 51 with value: 806.0024020395334.
[I 2026-10-09 08:56:20,496] Trial 52 finished with value: 813.5553337475259 and parameters: {'learning_rate': 0.04407639423342833, 'num_leaves': 67, 'min_child_samples': 76, 'subsample': 0.6264466641325293, 'colsample_bytree': 0.5847879887388773, 'n_estimators': 500}. Best is trial 51 with value: 806.0024020395334.
[I 2026-10-09 08:56:36,967] Trial 53 finished wi

272

## 3. Train

MAE mean becomes P50. Quantile trees are P10 and P90 only. Coverage scale uses the pre-refit trees on the last 21 train days, then the trees are refit on all train.

In [4]:
models, metrics = train_models(
    history,
    model_dir=OUT_DIR,
    features=DAY_V4_FEATURES,
    max_lead_hours=DAY_MAX_LEAD_HOURS,
    role="next_day",
    quantiles=TAIL_QUANTILES,
    n_estimators=int(mean_params["n_estimators"]),
    lgb_params=mean_params,
)
display(metrics)
print("v4 interval_scale", models.interval_scale)
print("prod interval_scale", prod.interval_scale)

,metric,quantile,value
0,pinball,0.1,186.9
1,pinball,0.9,192.3
2,pinball,0.5,419.0
3,mae,0.5,838.0
4,coverage_80,0.8,0.8
5,interval_scale,0.8,2.1


v4 interval_scale {'gb': 2.0595256686210632}
prod interval_scale {'gb': 2.079867720603943}


In [5]:
scale_mse = naive_scale_for_product(train, PRODUCT)
holdout_v4 = resample_to_mode(predict_quantiles(holdout, models), SHORT)
holdout_prod = resample_to_mode(predict_quantiles(holdout, prod), SHORT)


def holdout_tbl(frame):
    scored = as_board_scored(frame, actual_col="demand_mw")
    return overall_metrics(scored, scale_mse=scale_mse)


v4_hold = holdout_tbl(holdout_v4).assign(pack="v4")
prod_hold = holdout_tbl(holdout_prod).assign(pack="prod")
display(pd.concat([v4_hold, prod_hold], ignore_index=True))
print("holdout hourly MAE v4", float(v4_hold["mae"].iloc[0]))
print("holdout hourly MAE prod", float(prod_hold["mae"].iloc[0]))
print("holdout width v4", float(v4_hold["interval_width"].iloc[0]))
print("holdout width prod", float(prod_hold["interval_width"].iloc[0]))
print("holdout coverage v4", float(v4_hold["coverage_80"].iloc[0]))
print("holdout coverage prod", float(prod_hold["coverage_80"].iloc[0]))

,split,n,mae,rmse,mape,bias,pinball_10,pinball_50,pinball_90,coverage_80,interval_width,skill_vs_climatology,rmsse,wrmsse,pack
0,all,2546,818.5,"1,125.2",0.0,18.3,182.8,409.3,187.8,0.8,"2,794.7",0.0,0.4,0.4,v4
1,all,2546,843.2,"1,109.8",0.0,-96.2,194.2,421.6,188.7,0.8,"2,895.9",0.0,0.4,0.4,prod


holdout hourly MAE v4 818.5302902787704
holdout hourly MAE prod 843.2196249484064
holdout width v4 2794.6948210871474
holdout width prod 2895.87277598117
holdout coverage v4 0.8036135113904164
holdout coverage prod 0.8205027494108406


## 4. Live freeze — every board day

Issued vs production vs v4 on frozen midnight features. Lower MAE is better; coverage near 80% with a **smaller** width is the point of this pack.

In [6]:
def metrics_row(frame, label, climatology):
    actual_col = "actual_mw" if "actual_mw" in frame.columns else "demand_mw"
    scored = as_board_scored(frame, actual_col=actual_col, climatology=climatology)
    tbl = overall_metrics(scored, scale_mse=board_naive_scale(scored))
    tbl.insert(0, "pack", label)
    return tbl, scored

board = load_board()
context = load_context_actuals()
archive = _load_archive_weather()
joined = _pack_rows(DAY, board, context, day_days=None)
joined = _attach_frozen_and_weather(DAY, joined, archive)
joined = _ensure_calendar(joined)
joined = attach_lookback_weather(joined, archive, days=8)
joined = attach_lookback_demand(joined, context, days=8)
joined = _recompute_weather_features(joined)
ready = _ready_predict_frame(joined, models)

v4_days = predict_quantiles(ready, models)
prod_days = predict_quantiles(_ready_predict_frame(joined, prod), prod)
keep = joined["actual_mw"].notna() & joined["p50"].notna()
issued_tbl, _ = metrics_row(joined.loc[keep], "issued (board)", prod.climatology)
prod_tbl, _ = metrics_row(prod_days.loc[keep].assign(actual_mw=joined.loc[keep, "actual_mw"].to_numpy()), "prod freeze", prod.climatology)
v4_tbl, _ = metrics_row(v4_days.loc[keep].assign(actual_mw=joined.loc[keep, "actual_mw"].to_numpy()), "v4 freeze", models.climatology)
live = pd.concat([issued_tbl, prod_tbl, v4_tbl], ignore_index=True)
display(live)
print("n days", joined.loc[keep, "timestamp"].dt.tz_convert(LONDON_TZ).dt.normalize().nunique())

days = pd.DataFrame({
    "day": joined.loc[keep, "timestamp"].dt.tz_convert(LONDON_TZ).dt.strftime("%a %d %b"),
    "actual": pd.to_numeric(joined.loc[keep, "actual_mw"], errors="coerce"),
    "issued": pd.to_numeric(joined.loc[keep, "p50"], errors="coerce"),
    "prod": pd.to_numeric(prod_days.loc[keep, "p50"], errors="coerce"),
    "v4": pd.to_numeric(v4_days.loc[keep, "p50"], errors="coerce"),
    "issued_width": pd.to_numeric(joined.loc[keep, "p90"], errors="coerce") - pd.to_numeric(joined.loc[keep, "p10"], errors="coerce"),
    "prod_width": pd.to_numeric(prod_days.loc[keep, "p90"], errors="coerce") - pd.to_numeric(prod_days.loc[keep, "p10"], errors="coerce"),
    "v4_width": pd.to_numeric(v4_days.loc[keep, "p90"], errors="coerce") - pd.to_numeric(v4_days.loc[keep, "p10"], errors="coerce"),
})
by_day = days.groupby("day", sort=False).agg(
    n=("actual", "size"),
    issued_mae=("issued", lambda s: (days.loc[s.index, "actual"] - s).abs().mean()),
    prod_mae=("prod", lambda s: (days.loc[s.index, "actual"] - s).abs().mean()),
    v4_mae=("v4", lambda s: (days.loc[s.index, "actual"] - s).abs().mean()),
    issued_width=("issued_width", "mean"),
    prod_width=("prod_width", "mean"),
    v4_width=("v4_width", "mean"),
).reset_index()
display(by_day)

,pack,split,n,mae,rmse,mape,bias,pinball_10,pinball_50,pinball_90,coverage_80,interval_width,skill_vs_climatology,rmsse,wrmsse
0,issued (board),all,549,931.0,"1,284.2",0.0,-170.6,237.2,465.5,179.5,0.8,"2,959.7",0.5,0.6,0.6
1,prod freeze,all,549,918.7,"1,251.8",0.0,-392.0,221.5,459.3,188.4,0.9,"3,187.7",0.6,0.6,0.6
2,v4 freeze,all,549,936.1,"1,174.2",0.0,258.9,227.8,468.1,203.2,0.9,"3,741.9",0.5,0.6,0.5


n days 13


,day,n,issued_mae,prod_mae,v4_mae,issued_width,prod_width,v4_width
0,Sun 27 Sep,4,293.7,375.0,472.9,"2,694.6","3,162.8","3,371.6"
1,Mon 28 Sep,48,721.8,592.4,752.4,"2,968.7","2,907.4","3,492.1"
2,Tue 29 Sep,48,579.5,633.5,595.6,"2,723.2","3,244.3","3,492.0"
3,Wed 30 Sep,48,957.9,647.8,881.0,"2,844.1","3,273.4","3,796.7"
4,Thu 01 Oct,48,821.9,810.0,791.4,"3,040.9","3,297.0","3,804.0"
5,Fri 02 Oct,48,"1,437.7","1,302.9",658.3,"3,098.1","3,452.8","3,881.9"
6,Sat 03 Oct,48,663.9,628.1,"1,184.1","2,789.1","3,032.1","3,434.9"
7,Sun 04 Oct,48,"1,154.5","1,531.5",870.0,"2,994.5","3,102.4","3,713.6"
8,Mon 05 Oct,48,"1,689.2","1,947.9","1,102.0","2,837.2","2,941.8","4,114.2"
9,Tue 06 Oct,48,578.8,548.4,"1,360.7","2,913.9","2,956.1","3,449.2"


## 5. SHAP

`lag_48` should still dominate. The top-15 table hides weaker columns: a second table lists every 24h extra (`*_delta_1d`, `demand_lag_96`).

In [7]:
sample = v4_days.loc[keep].head(400)
extras = [c for c in DAY_V4_FEATURES if c not in FEATURE_COLUMNS]
if sample.empty:
    print("no scored rows for SHAP")
else:
    contrib = explain_scored(sample, models)["mean"]
    display(mean_abs_shap(contrib, n=20))
    extra_shap = (
        contrib.reindex(columns=extras)
        .abs()
        .mean()
        .sort_values(ascending=False)
        .rename("mean_abs_shap")
        .rename_axis("feature")
        .reset_index()
    )
    extra_shap["in_model"] = extra_shap["feature"].isin(models.features)
    display(extra_shap)
    display(shap_source_share(contrib))

,feature,mean_abs_shap
0,demand_lag_48,"1,899.5"
1,demand_lag_336,"1,084.1"
2,demand_lag_96,713.3
3,day_of_week,688.5
4,wind_speed_10m,291.2
5,settlement_period,277.2
6,period_sin,253.1
7,shortwave_radiation,219.3
8,is_weekend,190.0
9,temp_mean_fwd_12h,175.8


,source,mean_abs_shap,share
0,demand_lag,"3,712.9",0.5
1,weather,"1,845.5",0.2
2,calendar,"1,869.6",0.3
3,lead,28.7,0.0


## 6. Promote

Leave `False` until live MAE is lower **and** width is lower with coverage still near 80%.

In [8]:
PROMOTE = False
if PROMOTE:
    dest = ROOT / "data" / "models" / "day"
    dest.mkdir(parents=True, exist_ok=True)
    for name in ("gb_mean.txt", "gb_resid_q10.txt", "gb_resid_q50.txt", "gb_resid_q90.txt", "metadata.json", "metrics.csv", "climatology.parquet"):
        src = OUT_DIR / name
        if src.exists():
            shutil.copy2(src, dest / name)
    print("copied into", dest)
else:
    print("not promoted; artefacts in", OUT_DIR)

not promoted; artefacts in D:\energy-forecast\notebooks/day/v4\models
